In [1]:
import json

# Load files
with open(r"C:\Users\ysass\OneDrive\Desktop\Grad School Files\BSAN 6200 Text Mining and Social Media Analytics\text-analytics-a4-Sasson-Acosta\text-analytics-a4-Sasson-Acosta\data\raw\dataset_train.json") as f:
    data = [json.loads(line) for line in f]

with open(r"C:\Users\ysass\OneDrive\Desktop\Grad School Files\BSAN 6200 Text Mining and Social Media Analytics\text-analytics-a4-Sasson-Acosta\text-analytics-a4-Sasson-Acosta\data\raw\dataset_label.json") as f:
    label_map = json.load(f)

# Reverse label map
id_to_label = {v: k for k, v in label_map.items()}

def clean_label(label):
    return label.replace("B-", "").replace("I-", "")

output = []

for item in data:
    tokens = item["tokens"]
    tags = item["tags"]

    text = " ".join(tokens)
    spans = []

    char_pos = 0
    current_entity = None

    for token, tag in zip(tokens, tags):
        label = id_to_label[tag]

        start = char_pos
        end = char_pos + len(token)

        if label.startswith("B-"):
            if current_entity:
                spans.append(current_entity)

            current_entity = {
                "label": clean_label(label),
                "start": start,
                "end": end,
                "text": token
            }

        elif label.startswith("I-") and current_entity:
            current_entity["end"] = end
            current_entity["text"] += " " + token

        else:
            if current_entity:
                spans.append(current_entity)
                current_entity = None

        char_pos = end + 1  # +1 for space

    if current_entity:
        spans.append(current_entity)

    annotation = {
        "data": {"text": text},
        "annotations": [
            {
                "result": [
                    {
                        "from_name": "label",
                        "to_name": "text",
                        "type": "labels",
                        "value": {
                            "start": span["start"],
                            "end": span["end"],
                            "text": span["text"],
                            "labels": [span["label"]]
                        }
                    } for span in spans
                ]
            }
        ]
    }

    output.append(annotation)

# Save
with open("label_studio_import.json", "w") as f:
    json.dump(output, f, indent=2)

In [13]:
len(output)

6900